# Installations

In [ ]:

%pip install "pyautogen==0.5.3"

In [ ]:
%pip install azure-identity

In [ ]:
%pip install langchain-core langchain langchain-openai openai httpx

In [ ]:
%pip install --force-reinstall --no-cache-dir numpy pandas

In [ ]:
%pip install openpyxl

In [4]:
import json
import sys
from pathlib import Path
import pandas as pd


_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / "dc_available").is_dir()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path(r"c:\Users\VAS1UL\Master Thesis\DC-Avaialable Baseversion\dc-available")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


INGESTION_ROOT = REPO_ROOT / "dc_available" / "data" / "Artifacts" / "resources" / "ingestion"
OUTPUT_XLSX    = REPO_ROOT / "evaluation" / "dataset_generation" / "generated_dataset.xlsx"

from dc_available.config.llm_config import get_langchain_llm

QUESTION_MODEL        = "gpt-4o"  
QUESTION_TEMPERATURE  = 0.2
GENERATOR_MODEL       = "gpt-4o"   
GENERATOR_TEMPERATURE = 0.4

NUM_QUESTIONS     = 5
MAX_FILES         = None     

print("REPO_ROOT      :", REPO_ROOT)
print("INGESTION_ROOT :", INGESTION_ROOT, "(exists:", INGESTION_ROOT.exists(), ")")
print("OUTPUT_XLSX    :", OUTPUT_XLSX)


c:\Users\VAS1UL\pipx\venvs\poetry\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


REPO_ROOT      : c:\Users\VAS1UL\Master Thesis\DC-Avaialable Baseversion\dc-available
INGESTION_ROOT : c:\Users\VAS1UL\Master Thesis\DC-Avaialable Baseversion\dc-available\dc_available\data\Artifacts\resources\ingestion (exists: True )
OUTPUT_XLSX    : c:\Users\VAS1UL\Master Thesis\DC-Avaialable Baseversion\dc-available\evaluation\dataset_generation\generated_dataset.xlsx


In [5]:
def discover_jsonl_files(root: Path, max_files=None) -> list[Path]:
   
    if not root.exists():
        raise FileNotFoundError(f"Ingestion root does not exist: {root}")
    files = sorted(root.rglob("*__summary_docs.jsonl"))
    if max_files:
        files = files[:max_files]
    return files

# print discovered files
all_files = discover_jsonl_files(INGESTION_ROOT, max_files=MAX_FILES)
print(f"Found {len(all_files)} file(s):")
for f in all_files:
    print("  -", f.relative_to(INGESTION_ROOT))

# Sample File 
sample_file = all_files[0]
print("\nWalkthrough file:", sample_file.name)


Found 246 file(s):
  - DigitalTransformation\Basic\Training1\20221129_EN_eT-B-Digit_FINAL.pptx__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel1.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel10.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel11.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel12.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel13.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel14.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel15.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel16.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel17.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel18.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel19.mp4__summary_docs.jsonl
  - DigitalTransformation\Basic\Training1\Kapitel2

In [6]:
def parse_jsonl_file(path: Path) -> list[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"  ! parse error {path.name}:{line_num} -> {e}")
    return records


def extract_document_metadata(path: Path, records: list[dict]) -> dict:
    name = path.name.replace("__summary_docs.jsonl", "")

    # type: prefer an extension-like file_type from the records (".pptx", ".mp4", ".html")
    doc_type = ""
    for rec in records:
        ft = rec.get("metadata", {}).get("file_type", "")
        if isinstance(ft, str) and ft.startswith("."):
            doc_type = ft
            break
    if not doc_type:
        doc_type = Path(name).suffix or "unknown"

    parts = path.parts
    level = parts[-3] if len(parts) >= 3 else "unknown"
    topic = parts[-4] if len(parts) >= 4 else "unknown"

    return {"name": name, "type": doc_type, "topic": topic, "level": level}

# sample file
sample_records = parse_jsonl_file(sample_file)
sample_meta = extract_document_metadata(sample_file, sample_records)
print(f"Records: {len(sample_records)}")
for k, v in sample_meta.items():
    print(f"  {k:16}: {v}")

Records: 111
  name            : 20221129_EN_eT-B-Digit_FINAL.pptx
  type            : .png
  topic           : DigitalTransformation
  level           : Basic


In [7]:
import json
def clean_content(text: str) -> str:

    if not isinstance(text, str):
        return ""
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):      
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"): 
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    return text


def build_context(records: list[dict]) -> str:
   
    seen: set[str] = set()
    blocks: list[str] = []
    total = 0

    for rec in records:
        meta = rec.get("metadata", {})
        page_id  = meta.get("page_id")
        doc_type = meta.get("document_type", "")

        content   = clean_content(rec.get("page_content", ""))   
        meta_text = clean_content(meta.get("text", "") or "")    

        pieces = []
        for piece in (content, meta_text):
            piece = piece.strip()
            if not piece or piece in seen:   
                continue
            seen.add(piece)
            pieces.append(piece)
        if not pieces:
            continue

        blocks.append(f"[chunk page_id={page_id} type={doc_type}]\n" + "\n".join(pieces))

    return "\n\n".join(blocks)


sample_context = build_context(sample_records)
print(f"Context length: {len(sample_context)} chars\n")
print(sample_context)

Context length: 116727 chars

[chunk page_id=-1 type=pptx]
{
  "title": "Comprehensive Insights into Digital Transformation and the Factory of the Future",
  "description": "This collection of resources from the Bosch Rexroth Academy provides an in-depth exploration of digital transformation, Industry 4.0, and the Factory of the Future, with a focus on the evolution into Industry 5.0. It covers the historical context of industrial revolutions, key technologies like IoT, AI, digital twins, and cyber-physical systems, and strategies for integrating IT and OT to create smart, connected factories. The materials emphasize the transition from hierarchical automation pyramids to decentralized, interconnected systems, highlighting opportunities such as increased productivity, flexibility, and sustainability, alongside challenges like data quality, organizational silos, and ethical concerns. Key concepts include MES, ERP, human-machine collaboration, predictive maintenance, and the use of big d

In [8]:
QUESTION_INSTRUCTIONS = """
You are an expert at creating evaluation questions for a company knowledge assistant (RAG system).

You are given CONTEXT extracted from ONE training document. The text comes from JSON
chunk files; the original document may be a PowerPoint (.pptx), a PDF, an HTML web page,
or a video (slide screenshots + spoken transcript).

Document being processed:
- Name : <NAME>
- Type : <TYPE>
- Topic: <TOPIC>
- Level: <LEVEL>

Task: write exactly <N> questions that a learner or employee might type into a search or
chat assistant to learn about this topic.

RULES:
1. Natural language, as a real user would ask.
2. NEVER mention the document, slide, page, video, presentation, or "training material".
   Each question must stand on its own.
3. Every question must be answerable from the CONTEXT.
4. IGNORE low-value content: title slides, agenda / table of contents, section dividers,
   "thank you" / closing slides, contact details, legal notices, placeholders, or anything
   unrelated to the topic. Do not create questions from it.
5. QUESTION TYPES (use exactly one per question, one of each across the 5):
        1. factual     - Direct recall of a fact, definition, or specific data point
        2. conceptual  - Understanding of a concept, relationship, or principle
        3. procedural  - How something works or the steps in a process
        4. application - How a concept is applied in a real-world scenario
        5. summary     - Synthesis of multiple related ideas into one answer
6. The question should be of moderate difficulty.
7. The question should be framed from a part of context that contains important information.
8. The question should be fully answered from the given context.

To generate the questions, first identify the most important or relevant part of the context.
Then frame a question around that part that satisfies all the rules above.

Return ONLY a JSON array of objects, one per question, each with the question text and its type, e.g.
[{"question": "First question?", "question_type": "factual"}, {"question": "Second question?", "question_type": "conceptual"}]
The question_type must be exactly one of: factual, conceptual, procedural, application, summary.
No text before or after the JSON array.
"""


def parse_json_array(text: str):
    start, end = text.find("["), text.rfind("]") + 1
    if start < 0 or end <= start:
        raise ValueError(f"No JSON array found in response:\n{text[:300]}")
    return json.loads(text[start:end])


question_llm = get_langchain_llm(model_name=QUESTION_MODEL, temperature=QUESTION_TEMPERATURE)


def question_agent(context: str, meta: dict, n: int = NUM_QUESTIONS) -> list[dict]:
    prompt = (
        QUESTION_INSTRUCTIONS
        .replace("<N>", str(n))
        .replace("<NAME>", meta["name"])
        .replace("<TYPE>", meta["type"])
        .replace("<TOPIC>", meta["topic"])
        .replace("<LEVEL>", meta["level"])
        + "\n\nCONTEXT:\n" + context
    )
    response = question_llm.invoke(prompt)
    text = response.content if hasattr(response, "content") else str(response)
    items = parse_json_array(text)

    questions = []
    for it in items:
        if isinstance(it, dict):
            q = str(it.get("question", "")).strip()
            qtype = str(it.get("question_type", "")).strip()
        else:
            q, qtype = str(it).strip(), ""
        if q:
            questions.append({"question": q, "question_type": qtype})
    return questions


sample_questions = question_agent(sample_context, sample_meta)
for i, q in enumerate(sample_questions, 1):
    print(f"{i}. [{q['question_type']}] {q['question']}")


1. [factual] What is Industry 4.0 and how does it relate to digital transformation?
2. [conceptual] How does the transition from Industry 3.0 to Industry 4.0 impact the structure of the automation pyramid?
3. [procedural] What are the steps involved in implementing a Manufacturing Execution System (MES) in a factory?
4. [application] How can artificial intelligence be applied to improve quality control in production processes?
5. [summary] What are the key opportunities and challenges associated with Industry 4.0 and the Factory of the Future?


In [9]:
GENERATOR_INSTRUCTIONS = """You are an expert at writing correct, grounded answers for an evaluation dataset.

You are given CONTEXT extracted from one training document and a list of QUESTIONS
(each item has a "question" and its "question_type").
Only use the given context as a source for generating the answer
For EACH question, write an answer grounded ONLY in the CONTEXT. 
Be concise in answering the question and only answer the question at hand rather than adding extra information.

Copy these values exactly into every object:
- topic           = <TOPIC>
- source_document = <SOURCE>

Return ONLY a JSON array. Each element must be an object with exactly these keys:
{
  "question": "the question, copied exactly from the list",
  "question_type": "copied exactly from the same item in the list",
  "answer": "a correct answer grounded only in the context (1-3 sentences)",
  "keywords": ["3 to 5 short domain keywords"],
  "topic": "<TOPIC>",
  "source_document": "<SOURCE>"
}
No text before or after the JSON array."""


generator_llm = get_langchain_llm(model_name=GENERATOR_MODEL, temperature=GENERATOR_TEMPERATURE)


def generator_agent(context: str, questions: list[dict], meta: dict) -> list[dict]:
    prompt = (
        GENERATOR_INSTRUCTIONS
        .replace("<TOPIC>", meta["topic"])
        .replace("<SOURCE>", meta["name"])
        + "\n\nCONTEXT:\n" + context
        + "\n\nQUESTIONS:\n" + json.dumps(questions, ensure_ascii=False, indent=2)
    )
    response = generator_llm.invoke(prompt)
    text = response.content if hasattr(response, "content") else str(response)
    rows = parse_json_array(text)

    clean_rows = []
    for row in rows:
        if not isinstance(row, dict) or not row.get("answer"):
            continue
        kw = row.get("keywords", [])
        if isinstance(kw, list):
            kw = ", ".join(str(k) for k in kw)
        clean_rows.append({
            "topic":           meta["topic"],      # overwrite from metadata (ground truth)
            "level":           meta["level"],
            "type":            meta["type"],
            "source_document": meta["name"],
            "question":        row.get("question", ""),
            "question_type":   row.get("question_type", ""),
            "answer":          row.get("answer", ""),
            "keywords":        kw,
        })
    return clean_rows



sample_rows = generator_agent(sample_context, sample_questions, sample_meta)
df = pd.DataFrame(sample_rows)
df

,topic,level,type,source_document,question,question_type,answer,keywords
0,DigitalTransformation,Basic,.png,20221129_EN_eT-B-Digit_FINAL.pptx,What is Industry 4.0 and how does it relate to...,factual,Industry 4.0 refers to the integration of peop...,"Industry 4.0, digital transformation, IoT, AI,..."
1,DigitalTransformation,Basic,.png,20221129_EN_eT-B-Digit_FINAL.pptx,How does the transition from Industry 3.0 to I...,conceptual,The transition from Industry 3.0 to Industry 4...,"Industry 3.0, Industry 4.0, automation pyramid..."
2,DigitalTransformation,Basic,.png,20221129_EN_eT-B-Digit_FINAL.pptx,What are the steps involved in implementing a ...,procedural,Implementing an MES involves connecting it to ...,"MES, Manufacturing Execution System, data acqu..."
3,DigitalTransformation,Basic,.png,20221129_EN_eT-B-Digit_FINAL.pptx,How can artificial intelligence be applied to ...,application,Artificial intelligence can improve quality co...,"artificial intelligence, quality control, prod..."
4,DigitalTransformation,Basic,.png,20221129_EN_eT-B-Digit_FINAL.pptx,What are the key opportunities and challenges ...,summary,"Opportunities include increased productivity, ...","Industry 4.0, Factory of the Future, opportuni..."


In [ ]:
# convert df into excel
df.to_excel('test.xlsx', index=False, engine="openpyxl")

In [10]:
all_rows: list[dict] = []

for path in all_files:
    try:
        records = parse_jsonl_file(path)
        if not records:
            print(f"  skip  : {path.name} (no records)")
            continue

        meta = extract_document_metadata(path, records)
        context = build_context(records)
        if not context:
            print(f"  skip  : {path.name} (no usable content)")
            continue

        questions = question_agent(context, meta)
        rows = generator_agent(context, questions, meta)
        all_rows.extend(rows)
        print(f"  ok    : {path.name} ({len(rows)} rows)")
    except Exception as e:
        print(f"  error : {path.name} -> {e}")

df = pd.DataFrame(all_rows, columns=[
    "topic", "level", "type", "source_document",
    "question", "question_type", "answer", "keywords",
])


OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
df.to_excel(OUTPUT_XLSX, index=False, engine="openpyxl")
print(f"\nSaved {len(df)} rows -> {OUTPUT_XLSX}")
df.head(20)


  ok    : 20221129_EN_eT-B-Digit_FINAL.pptx__summary_docs.jsonl (5 rows)
  ok    : Kapitel1.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel10.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel11.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel12.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel13.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel14.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel15.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel16.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel17.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel18.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel19.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel2.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel20.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel3.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel4.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel5.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel6.mp4__summary_docs.jsonl (5 rows)
  ok    : Kapitel7.mp4__summary_

,topic,level,type,source_document,question,question_type,answer,keywords
0,DigitalTransformation,Basic,.png,20221129_EN_eT-B-Digit_FINAL.pptx,What are the key technologies driving Industry...,factual,Key technologies driving Industry 4.0 include ...,"IoT, AI, digital twins, cyber-physical systems..."
1,DigitalTransformation,Basic,.png,20221129_EN_eT-B-Digit_FINAL.pptx,How does the transition from Industry 3.0 to I...,conceptual,The transition from Industry 3.0 to Industry 4...,"Industry 3.0, Industry 4.0, decentralization, ..."
2,DigitalTransformation,Basic,.png,20221129_EN_eT-B-Digit_FINAL.pptx,What are the steps involved in implementing a ...,procedural,Implementing digital transformation involves s...,"digital transformation, connectivity, IT-OT in..."
3,DigitalTransformation,Basic,.png,20221129_EN_eT-B-Digit_FINAL.pptx,How can augmented reality be applied to improv...,application,Augmented reality can improve productivity in ...,"augmented reality, productivity, real-time vis..."
4,DigitalTransformation,Basic,.png,20221129_EN_eT-B-Digit_FINAL.pptx,What are the main opportunities and challenges...,summary,Opportunities of Industry 4.0 include increase...,"Industry 4.0, opportunities, challenges, produ..."
5,DigitalTransformation,Basic,.mp4,Kapitel1.mp4,What are the four stages of industrial revolut...,factual,The four stages of industrial revolutions are:...,"Industry 4.0, industrial revolutions, mechaniz..."
6,DigitalTransformation,Basic,.mp4,Kapitel1.mp4,How has the concept of Industry 4.0 transforme...,conceptual,Industry 4.0 has transformed factories by intr...,"Industry 4.0, smart factories, IoT, connectivi..."
7,DigitalTransformation,Basic,.mp4,Kapitel1.mp4,What are the key triggers driving the need for...,procedural,The key triggers driving Industry 4.0 and digi...,"Industry 4.0, digital transformation, triggers..."
8,DigitalTransformation,Basic,.mp4,Kapitel1.mp4,How can adopting a 'Think in Data' mindset hel...,application,Adopting a 'Think in Data' mindset helps busin...,"Think in Data, volatile markets, data-driven, ..."
9,DigitalTransformation,Basic,.mp4,Kapitel1.mp4,Can you summarize the societal impacts of the ...,summary,The societal impacts of the industrial revolut...,"industrial revolutions, societal impacts, conn..."
